In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import xgboost

print("Ambiente configurado com sucesso!")

Ambiente configurado com sucesso!


In [6]:
import pandas as pd

path = "data/"

base_cadastral = pd.read_csv(
    path + "base_cadastral.csv",
    sep=";"
)

base_info = pd.read_csv(
    path + "base_info.csv",
    sep=";"
)

base_pagamentos_dev = pd.read_csv(
    path + "base_pagamentos_desenvolvimento.csv",
    sep=";"
)

base_pagamentos_teste = pd.read_csv(
    path + "base_pagamentos_teste.csv",
    sep=";"
)

print("Bases carregadas corretamente!")

Bases carregadas corretamente!


In [7]:
print("Base cadastral:", base_cadastral.shape)
print("Base info:", base_info.shape)
print("Base pagamentos desenvolvimento:", base_pagamentos_dev.shape)
print("Base pagamentos teste:", base_pagamentos_teste.shape)

Base cadastral: (1315, 8)
Base info: (24401, 4)
Base pagamentos desenvolvimento: (77414, 7)
Base pagamentos teste: (12275, 6)


In [11]:
base_cadastral.head()

,ID_CLIENTE,DATA_CADASTRO,DDD,FLAG_PF,SEGMENTO_INDUSTRIAL,DOMINIO_EMAIL,PORTE,CEP_2_DIG
0,1661240395903230676,2013-08-22,99,NaN,Serviços,YAHOO,PEQUENO,65
1,8274986328479596038,2017-01-25,31,NaN,Comércio,YAHOO,MEDIO,77
2,345447888460137901,2000-08-15,75,NaN,Serviços,HOTMAIL,PEQUENO,48
3,1003144834589372198,2017-08-06,49,NaN,Serviços,OUTLOOK,PEQUENO,89
4,324916756972236008,2011-02-14,88,NaN,Serviços,GMAIL,GRANDE,62


In [8]:
base_info.head()

,ID_CLIENTE,SAFRA_REF,RENDA_MES_ANTERIOR,NO_FUNCIONARIOS
0,1661240395903230676,2018-09,16913.0,NaN
1,8274986328479596038,2018-09,106430.0,141.0
2,345447888460137901,2018-09,707439.0,99.0
3,1003144834589372198,2018-09,239659.0,96.0
4,324916756972236008,2018-09,203123.0,103.0


In [9]:
base_pagamentos_dev.head()

,ID_CLIENTE,SAFRA_REF,DATA_EMISSAO_DOCUMENTO,DATA_PAGAMENTO,DATA_VENCIMENTO,VALOR_A_PAGAR,TAXA
0,1661240395903230676,2018-08,2018-08-17,2018-09-06,2018-09-06,35516.41,6.99
1,1661240395903230676,2018-08,2018-08-19,2018-09-11,2018-09-10,17758.21,6.99
2,1661240395903230676,2018-08,2018-08-26,2018-09-18,2018-09-17,17431.96,6.99
3,1661240395903230676,2018-08,2018-08-30,2018-10-11,2018-10-05,1341.00,6.99
4,1661240395903230676,2018-08,2018-08-31,2018-09-20,2018-09-20,21309.85,6.99


In [10]:
base_pagamentos_teste.head()

,ID_CLIENTE,SAFRA_REF,DATA_EMISSAO_DOCUMENTO,DATA_VENCIMENTO,VALOR_A_PAGAR,TAXA
0,5058298901476893676,2021-07,2021-07-14,2021-08-04,11204.75,4.99
1,274692171162531764,2021-07,2021-07-08,2021-08-23,60718.50,5.99
2,274692171162531764,2021-07,2021-07-11,2021-08-25,60718.50,5.99
3,274692171162531764,2021-07,2021-07-16,2021-08-30,62250.00,5.99
4,465309249432033993,2021-07,2021-07-05,2021-07-30,26593.95,6.99


In [12]:
# Converter datas da base cadastral

base_cadastral["DATA_CADASTRO"] = pd.to_datetime(
    base_cadastral["DATA_CADASTRO"]
)

In [13]:
# Converter datas da base de pagamentos

colunas_datas = [
    "DATA_EMISSAO_DOCUMENTO",
    "DATA_PAGAMENTO",
    "DATA_VENCIMENTO"
]

for coluna in colunas_datas:
    base_pagamentos_dev[coluna] = pd.to_datetime(
        base_pagamentos_dev[coluna]
    )

In [14]:
base_pagamentos_dev.dtypes

ID_CLIENTE                         int64
SAFRA_REF                            str
DATA_EMISSAO_DOCUMENTO    datetime64[us]
DATA_PAGAMENTO            datetime64[us]
DATA_VENCIMENTO           datetime64[us]
VALOR_A_PAGAR                    float64
TAXA                             float64
dtype: object

In [15]:
base_pagamentos_dev["DIAS_ATRASO"] = (
    base_pagamentos_dev["DATA_PAGAMENTO"] -
    base_pagamentos_dev["DATA_VENCIMENTO"]
).dt.days

In [16]:
base_pagamentos_dev[[
    "DATA_PAGAMENTO",
    "DATA_VENCIMENTO",
    "DIAS_ATRASO"
]].head(10)

,DATA_PAGAMENTO,DATA_VENCIMENTO,DIAS_ATRASO
0,2018-09-06,2018-09-06,0
1,2018-09-11,2018-09-10,1
2,2018-09-18,2018-09-17,1
3,2018-10-11,2018-10-05,6
4,2018-09-20,2018-09-20,0
5,2018-09-25,2018-09-25,0
6,2018-09-05,2018-09-05,0
7,2018-09-03,2018-09-03,0
8,2018-09-03,2018-09-03,0
9,2018-09-05,2018-09-05,0


In [17]:
base_pagamentos_dev["TARGET"] = (
    base_pagamentos_dev["DIAS_ATRASO"] >= 5
).astype(int)

In [18]:
base_pagamentos_dev[[
    "DIAS_ATRASO",
    "TARGET"
]].head(15)

,DIAS_ATRASO,TARGET
0,0,0
1,1,0
2,1,0
3,6,1
4,0,0
5,0,0
6,0,0
7,0,0
8,0,0
9,0,0


In [19]:
base_pagamentos_dev["TARGET"].value_counts()

TARGET
0    71978
1     5436
Name: count, dtype: int64

In [20]:
base_pagamentos_dev["TARGET"].value_counts(normalize=True)

TARGET
0    0.92978
1    0.07022
Name: proportion, dtype: float64

### Construção da variável target

A variável target foi criada conforme a regra de negócio definida no desafio:

- TARGET = 1: pagamentos realizados com atraso igual ou superior a 5 dias.
- TARGET = 0: pagamentos realizados com atraso inferior a 5 dias.

Após a criação, observou-se que a base possui aproximadamente 7% de casos de inadimplência, caracterizando um problema de classificação desbalanceado.

In [21]:
base_pagamentos_dev.isnull().sum()

ID_CLIENTE                   0
SAFRA_REF                    0
DATA_EMISSAO_DOCUMENTO       0
DATA_PAGAMENTO               0
DATA_VENCIMENTO              0
VALOR_A_PAGAR             1170
TAXA                         0
DIAS_ATRASO                  0
TARGET                       0
dtype: int64

In [22]:
base_pagamentos_dev["VALOR_A_PAGAR"].median()

np.float64(34758.695)

In [23]:
base_pagamentos_dev["VALOR_A_PAGAR"].describe()

count    7.624400e+04
mean     4.659078e+04
std      4.643393e+04
min      1.000000e-01
25%      1.876536e+04
50%      3.475869e+04
75%      6.093384e+04
max      4.400000e+06
Name: VALOR_A_PAGAR, dtype: float64

In [24]:
base_pagamentos_dev.sort_values(
    "VALOR_A_PAGAR",
    ascending=False
).head(10)


,ID_CLIENTE,SAFRA_REF,DATA_EMISSAO_DOCUMENTO,DATA_PAGAMENTO,DATA_VENCIMENTO,VALOR_A_PAGAR,TAXA,DIAS_ATRASO,TARGET
45230,8874257794774679816,2020-04,2020-04-21,2020-06-25,2020-06-03,4400000.00,6.99,22,1
8424,5918810537171565818,2018-12,2018-12-19,2019-04-02,2019-01-31,2250000.00,5.99,61,1
10748,5761480994209806499,2019-01,2019-01-21,2019-02-04,2019-01-25,1697544.07,5.99,10,1
42467,7204322229254419755,2020-03,2020-03-11,2020-05-06,2020-04-13,1500000.00,5.99,23,1
15995,1366416810235233769,2019-03,2019-03-27,2019-04-11,2022-02-28,1391835.20,6.99,-1054,0
14263,7204322229254419755,2019-02,2019-02-25,2019-05-02,2019-04-16,1325000.00,4.99,16,1
48878,5999145415800242177,2020-06,2020-06-18,2021-02-10,2020-07-31,1210000.00,4.99,194,1
32639,4569866309119931805,2019-10,2019-10-26,2020-01-27,2019-12-08,1200000.00,4.99,50,1
12985,7751080728852739823,2019-03,2019-03-04,2019-03-19,2019-04-30,1160000.00,6.99,-42,0
66292,6784879562354696650,2021-02,2021-02-02,2021-02-17,2024-06-30,1000000.00,5.99,-1229,0


In [25]:
(base_pagamentos_dev["DIAS_ATRASO"] < 0).sum()

np.int64(7906)

In [26]:
base_pagamentos_dev[
    base_pagamentos_dev["DIAS_ATRASO"] < 0
]["DIAS_ATRASO"].describe()

count    7906.000000
mean      -16.047812
std        69.119894
min     -2661.000000
25%       -13.000000
50%        -3.000000
75%        -1.000000
max        -1.000000
Name: DIAS_ATRASO, dtype: float64

In [27]:
base_pagamentos_dev["DIAS_ATRASO_AJUSTADO"] = (
    base_pagamentos_dev["DIAS_ATRASO"]
    .clip(lower=0)
)

base_pagamentos_dev[
    ["DIAS_ATRASO", "DIAS_ATRASO_AJUSTADO"]
].head(15)

,DIAS_ATRASO,DIAS_ATRASO_AJUSTADO
0,0,0
1,1,1
2,1,1
3,6,6
4,0,0
5,0,0
6,0,0
7,0,0
8,0,0
9,0,0


In [28]:
base_pagamentos_dev["TARGET"] = (
    base_pagamentos_dev["DIAS_ATRASO_AJUSTADO"] >= 5
).astype(int)

In [29]:
base_pagamentos_dev["TARGET"].value_counts()

TARGET
0    71978
1     5436
Name: count, dtype: int64

In [30]:
base_pagamentos_dev["TARGET"].value_counts(normalize=True)

TARGET
0    0.92978
1    0.07022
Name: proportion, dtype: float64

In [31]:
base_pagamentos_dev.describe()

,ID_CLIENTE,DATA_EMISSAO_DOCUMENTO,DATA_PAGAMENTO,DATA_VENCIMENTO,VALOR_A_PAGAR,TAXA,DIAS_ATRASO,TARGET,DIAS_ATRASO_AJUSTADO
count,7.741400e+04,77414,77414,77414,7.624400e+04,77414.000000,77414.000000,77414.000000,77414.000000
mean,4.662270e+18,2020-02-02 21:01:42.400082,2020-02-26 00:36:28.627379,2020-02-26 04:43:20.087839,4.659078e+04,6.789623,-0.171429,0.070220,1.467474
min,8.784237e+15,2018-08-17 00:00:00,2018-06-19 00:00:00,2017-11-27 00:00:00,1.000000e-01,4.990000,-2661.000000,0.000000,0.000000
25%,2.369365e+18,2019-05-21 00:00:00,2019-06-13 00:00:00,2019-06-13 00:00:00,1.876536e+04,5.990000,0.000000,0.000000,0.000000
50%,4.817817e+18,2020-01-27 00:00:00,2020-02-19 00:00:00,2020-02-18 00:00:00,3.475869e+04,5.990000,0.000000,0.000000,0.000000
75%,6.969349e+18,2020-10-27 00:00:00,2020-11-18 00:00:00,2020-11-18 00:00:00,6.093384e+04,6.990000,0.000000,0.000000,0.000000
max,9.206031e+18,2021-06-30 00:00:00,2021-11-24 00:00:00,2027-03-31 00:00:00,4.400000e+06,11.990000,869.000000,1.000000,869.000000
std,2.665719e+18,NaN,NaN,NaN,4.643393e+04,1.798225,25.229477,0.255519,10.965514


In [32]:
base_pagamentos_dev["ATRASO_BIN"] = pd.cut(
    base_pagamentos_dev["DIAS_ATRASO_AJUSTADO"],
    bins=[-1, 0, 5, 15, 30, 60, 9999],
    labels=[
        "EM_DIA",
        "1_5_DIAS",
        "6_15_DIAS",
        "16_30_DIAS",
        "31_60_DIAS",
        "60_PLUS"
    ]
)

base_pagamentos_dev["ATRASO_BIN"].value_counts()

ATRASO_BIN
EM_DIA        68648
1_5_DIAS       4285
6_15_DIAS      2997
16_30_DIAS      782
60_PLUS         365
31_60_DIAS      337
Name: count, dtype: int64

In [33]:
base_modelo = base_pagamentos_dev.merge(
    base_cadastral,
    on="ID_CLIENTE",
    how="left"
)

base_modelo.shape

(77414, 18)

In [34]:
base_modelo = base_modelo.merge(
    base_info,
    on="ID_CLIENTE",
    how="left"
)

base_modelo.shape

(2646642, 21)

In [35]:
base_info["ID_CLIENTE"].nunique()

1336

In [36]:
base_info.shape

(24401, 4)

In [37]:
base_info["ID_CLIENTE"].value_counts().head(10)

ID_CLIENTE
324916756972236008     40
4679462479444735708    40
8123515208041701633    40
3118152790263512426    40
2092588065958607289    40
1683619849073499603    40
8784237149961904       40
4347826483121154490    40
2397171153213063761    40
2634859679238614382    40
Name: count, dtype: int64

In [38]:
base_modelo = base_pagamentos_dev.merge(
    base_cadastral,
    on="ID_CLIENTE",
    how="left"
)

base_modelo.shape

(77414, 18)

In [39]:
base_info["SAFRA_REF"] = pd.to_datetime(
    base_info["SAFRA_REF"]
)

In [40]:
base_info.dtypes

ID_CLIENTE                     int64
SAFRA_REF             datetime64[us]
RENDA_MES_ANTERIOR           float64
NO_FUNCIONARIOS              float64
dtype: object

In [41]:
base_modelo = base_modelo.sort_values(
    ["ID_CLIENTE", "DATA_EMISSAO_DOCUMENTO"]
)

base_info = base_info.sort_values(
    ["ID_CLIENTE", "SAFRA_REF"]
)

In [42]:
base_modelo.shape, base_info.shape

((77414, 18), (24401, 4))

In [43]:
base_modelo = base_modelo.sort_values(
    "DATA_EMISSAO_DOCUMENTO"
)

base_info = base_info.sort_values(
    "SAFRA_REF"
)

In [44]:
base_modelo.shape, base_info.shape

((77414, 18), (24401, 4))

In [45]:
base_info["ID_CLIENTE"].duplicated().sum()

np.int64(23065)

In [46]:
base_info["ID_CLIENTE"].value_counts().head(10)

ID_CLIENTE
8784237149961904       40
6916556752218696485    40
3118152790263512426    40
6987942592445950913    40
6969348581033924947    40
3095686599956852697    40
3285246914299291124    40
3355881107559250653    40
6582676833651678345    40
459290606780222306     40
Name: count, dtype: int64

In [47]:
base_info["SAFRA_REF"].min(), base_info["SAFRA_REF"].max()

(Timestamp('2018-09-01 00:00:00'), Timestamp('2021-12-01 00:00:00'))

In [48]:
base_modelo[["ID_CLIENTE", "DATA_EMISSAO_DOCUMENTO", "SAFRA_REF"]].head()

,ID_CLIENTE,DATA_EMISSAO_DOCUMENTO,SAFRA_REF
166,8784237149961904,2018-08-17,2018-08
55,3118152790263512426,2018-08-17,2018-08
145,4717593620120611957,2018-08-17,2018-08
21,4679462479444735708,2018-08-17,2018-08
76,1366416810235233769,2018-08-17,2018-08


In [49]:
base_modelo["SAFRA_REF"] = pd.to_datetime(base_modelo["SAFRA_REF"])
base_info["SAFRA_REF"] = pd.to_datetime(base_info["SAFRA_REF"])

In [50]:
base_modelo["SAFRA_REF"].dtype, base_info["SAFRA_REF"].dtype

(dtype('<M8[us]'), dtype('<M8[us]'))

In [51]:
base_modelo = base_modelo.sort_values(
    ["ID_CLIENTE", "SAFRA_REF"]
)

base_info = base_info.sort_values(
    ["ID_CLIENTE", "SAFRA_REF"]
)

In [52]:
base_modelo.shape, base_info.shape

((77414, 18), (24401, 4))

In [53]:
base_modelo = pd.merge_asof(
    base_modelo,
    base_info,
    on="SAFRA_REF",
    by="ID_CLIENTE",
    direction="backward"
)

ValueError: left keys must be sorted

In [54]:
base_modelo = base_modelo.sort_values("SAFRA_REF")
base_info = base_info.sort_values("SAFRA_REF")

In [55]:
base_modelo["SAFRA_REF"].is_monotonic_increasing, base_info["SAFRA_REF"].is_monotonic_increasing

(True, True)

In [56]:
base_modelo = pd.merge_asof(
    base_modelo,
    base_info,
    on="SAFRA_REF",
    by="ID_CLIENTE",
    direction="backward"
)

In [57]:
base_modelo.shape

(77414, 20)

In [58]:
base_modelo = base_modelo.sort_values(
    ["ID_CLIENTE", "SAFRA_REF"]
).reset_index(drop=True)

base_info = base_info.sort_values(
    ["ID_CLIENTE", "SAFRA_REF"]
).reset_index(drop=True)

In [59]:
print(base_modelo[["ID_CLIENTE","SAFRA_REF"]].head())
print(base_info[["ID_CLIENTE","SAFRA_REF"]].head())

         ID_CLIENTE  SAFRA_REF
0  8784237149961904 2018-08-01
1  8784237149961904 2018-08-01
2  8784237149961904 2018-08-01
3  8784237149961904 2018-08-01
4  8784237149961904 2018-08-01
         ID_CLIENTE  SAFRA_REF
0  8784237149961904 2018-09-01
1  8784237149961904 2018-10-01
2  8784237149961904 2018-11-01
3  8784237149961904 2018-12-01
4  8784237149961904 2019-01-01


In [60]:
base_modelo = pd.merge_asof(
    base_modelo,
    base_info,
    on="SAFRA_REF",
    by="ID_CLIENTE",
    direction="backward"
)

ValueError: left keys must be sorted

In [61]:
base_modelo = base_modelo.sort_values(
    ["SAFRA_REF", "ID_CLIENTE"]
).reset_index(drop=True)

base_info = base_info.sort_values(
    ["SAFRA_REF", "ID_CLIENTE"]
).reset_index(drop=True)

In [62]:
print(base_modelo[["SAFRA_REF", "ID_CLIENTE"]].head())
print(base_info[["SAFRA_REF", "ID_CLIENTE"]].head())

   SAFRA_REF        ID_CLIENTE
0 2018-08-01  8784237149961904
1 2018-08-01  8784237149961904
2 2018-08-01  8784237149961904
3 2018-08-01  8784237149961904
4 2018-08-01  8784237149961904
   SAFRA_REF         ID_CLIENTE
0 2018-09-01   8784237149961904
1 2018-09-01  39547025441582855
2 2018-09-01  49632905576538927
3 2018-09-01  66220087398241662
4 2018-09-01  86110062990790869


In [63]:
base_modelo = pd.merge_asof(
    base_modelo,
    base_info,
    on="SAFRA_REF",
    by="ID_CLIENTE",
    direction="backward"
)

In [64]:
base_modelo.shape

(77414, 22)

In [65]:
base_modelo.head()

,ID_CLIENTE,SAFRA_REF,DATA_EMISSAO_DOCUMENTO,DATA_PAGAMENTO,DATA_VENCIMENTO,VALOR_A_PAGAR,TAXA,DIAS_ATRASO,TARGET,DIAS_ATRASO_AJUSTADO,...,DDD,FLAG_PF,SEGMENTO_INDUSTRIAL,DOMINIO_EMAIL,PORTE,CEP_2_DIG,RENDA_MES_ANTERIOR_x,NO_FUNCIONARIOS_x,RENDA_MES_ANTERIOR_y,NO_FUNCIONARIOS_y
0,8784237149961904,2018-08-01,2018-08-17,2018-09-04,2018-09-04,100616.1,5.99,0,0,0,...,11,NaN,Comércio,HOTMAIL,PEQUENO,27,NaN,NaN,NaN,NaN
1,8784237149961904,2018-08-01,2018-08-22,2018-09-11,2018-09-11,89552.8,5.99,0,0,0,...,11,NaN,Comércio,HOTMAIL,PEQUENO,27,NaN,NaN,NaN,NaN
2,8784237149961904,2018-08-01,2018-08-23,2018-09-08,2018-09-10,102686.1,5.99,-2,0,0,...,11,NaN,Comércio,HOTMAIL,PEQUENO,27,NaN,NaN,NaN,NaN
3,8784237149961904,2018-08-01,2018-08-23,2018-09-10,2018-09-10,94062.8,5.99,0,0,0,...,11,NaN,Comércio,HOTMAIL,PEQUENO,27,NaN,NaN,NaN,NaN
4,8784237149961904,2018-08-01,2018-08-24,2018-09-11,2018-09-11,51393.0,5.99,0,0,0,...,11,NaN,Comércio,HOTMAIL,PEQUENO,27,NaN,NaN,NaN,NaN


In [66]:
base_modelo[
    [
        "RENDA_MES_ANTERIOR_x",
        "RENDA_MES_ANTERIOR_y",
        "NO_FUNCIONARIOS_x",
        "NO_FUNCIONARIOS_y"
    ]
].isna().sum()

RENDA_MES_ANTERIOR_x    3717
RENDA_MES_ANTERIOR_y    3717
NO_FUNCIONARIOS_x       5265
NO_FUNCIONARIOS_y       5265
dtype: int64

In [67]:
(base_modelo["RENDA_MES_ANTERIOR_x"] == base_modelo["RENDA_MES_ANTERIOR_y"]).sum()

np.int64(73697)

In [68]:
(base_modelo["NO_FUNCIONARIOS_x"] == base_modelo["NO_FUNCIONARIOS_y"]).sum()

np.int64(72149)

In [69]:
import pandas as pd

print(
    base_modelo["RENDA_MES_ANTERIOR_x"].fillna(-1).equals(
        base_modelo["RENDA_MES_ANTERIOR_y"].fillna(-1)
    )
)

print(
    base_modelo["NO_FUNCIONARIOS_x"].fillna(-1).equals(
        base_modelo["NO_FUNCIONARIOS_y"].fillna(-1)
    )
)

True
True


In [70]:
base_modelo = base_modelo.drop(
    columns=[
        "RENDA_MES_ANTERIOR_y",
        "NO_FUNCIONARIOS_y"
    ]
)

base_modelo = base_modelo.rename(
    columns={
        "RENDA_MES_ANTERIOR_x": "RENDA_MES_ANTERIOR",
        "NO_FUNCIONARIOS_x": "NO_FUNCIONARIOS"
    }
)

In [71]:
base_modelo.shape

(77414, 20)

In [72]:
base_modelo.columns

Index(['ID_CLIENTE', 'SAFRA_REF', 'DATA_EMISSAO_DOCUMENTO', 'DATA_PAGAMENTO',
       'DATA_VENCIMENTO', 'VALOR_A_PAGAR', 'TAXA', 'DIAS_ATRASO', 'TARGET',
       'DIAS_ATRASO_AJUSTADO', 'ATRASO_BIN', 'DATA_CADASTRO', 'DDD', 'FLAG_PF',
       'SEGMENTO_INDUSTRIAL', 'DOMINIO_EMAIL', 'PORTE', 'CEP_2_DIG',
       'RENDA_MES_ANTERIOR', 'NO_FUNCIONARIOS'],
      dtype='str')

In [73]:
base_modelo.isna().sum()

ID_CLIENTE                    0
SAFRA_REF                     0
DATA_EMISSAO_DOCUMENTO        0
DATA_PAGAMENTO                0
DATA_VENCIMENTO               0
VALOR_A_PAGAR              1170
TAXA                          0
DIAS_ATRASO                   0
TARGET                        0
DIAS_ATRASO_AJUSTADO          0
ATRASO_BIN                    0
DATA_CADASTRO                 0
DDD                        7414
FLAG_PF                   77195
SEGMENTO_INDUSTRIAL        1417
DOMINIO_EMAIL               898
PORTE                      2476
CEP_2_DIG                     0
RENDA_MES_ANTERIOR         3717
NO_FUNCIONARIOS            5265
dtype: int64

In [74]:
base_modelo["VALOR_A_PAGAR"] = base_modelo["VALOR_A_PAGAR"].fillna(
    base_modelo["VALOR_A_PAGAR"].median()
)

In [75]:
base_modelo["RENDA_MES_ANTERIOR"] = base_modelo["RENDA_MES_ANTERIOR"].fillna(
    base_modelo["RENDA_MES_ANTERIOR"].median()
)

In [76]:
base_modelo["NO_FUNCIONARIOS"] = base_modelo["NO_FUNCIONARIOS"].fillna(
    base_modelo["NO_FUNCIONARIOS"].median()
)

In [77]:
base_modelo["DDD"] = base_modelo["DDD"].fillna("DESCONHECIDO")

In [78]:
base_modelo = base_modelo.drop(columns=["FLAG_PF"])

In [79]:
base_modelo["VALOR_A_PAGAR"] = base_modelo["VALOR_A_PAGAR"].fillna(
    base_modelo["VALOR_A_PAGAR"].median()
)

base_modelo["RENDA_MES_ANTERIOR"] = base_modelo["RENDA_MES_ANTERIOR"].fillna(
    base_modelo["RENDA_MES_ANTERIOR"].median()
)

base_modelo["NO_FUNCIONARIOS"] = base_modelo["NO_FUNCIONARIOS"].fillna(
    base_modelo["NO_FUNCIONARIOS"].median()
)

In [80]:
base_modelo.isna().sum()

ID_CLIENTE                   0
SAFRA_REF                    0
DATA_EMISSAO_DOCUMENTO       0
DATA_PAGAMENTO               0
DATA_VENCIMENTO              0
VALOR_A_PAGAR                0
TAXA                         0
DIAS_ATRASO                  0
TARGET                       0
DIAS_ATRASO_AJUSTADO         0
ATRASO_BIN                   0
DATA_CADASTRO                0
DDD                          0
SEGMENTO_INDUSTRIAL       1417
DOMINIO_EMAIL              898
PORTE                     2476
CEP_2_DIG                    0
RENDA_MES_ANTERIOR           0
NO_FUNCIONARIOS              0
dtype: int64

In [81]:
base_modelo["SEGMENTO_INDUSTRIAL"] = base_modelo["SEGMENTO_INDUSTRIAL"].fillna("DESCONHECIDO")

base_modelo["DOMINIO_EMAIL"] = base_modelo["DOMINIO_EMAIL"].fillna("DESCONHECIDO")

base_modelo["PORTE"] = base_modelo["PORTE"].fillna("DESCONHECIDO")

In [82]:
base_modelo.isnull().sum()

ID_CLIENTE                0
SAFRA_REF                 0
DATA_EMISSAO_DOCUMENTO    0
DATA_PAGAMENTO            0
DATA_VENCIMENTO           0
VALOR_A_PAGAR             0
TAXA                      0
DIAS_ATRASO               0
TARGET                    0
DIAS_ATRASO_AJUSTADO      0
ATRASO_BIN                0
DATA_CADASTRO             0
DDD                       0
SEGMENTO_INDUSTRIAL       0
DOMINIO_EMAIL             0
PORTE                     0
CEP_2_DIG                 0
RENDA_MES_ANTERIOR        0
NO_FUNCIONARIOS           0
dtype: int64

In [83]:
base_modelo.dtypes

ID_CLIENTE                         int64
SAFRA_REF                 datetime64[us]
DATA_EMISSAO_DOCUMENTO    datetime64[us]
DATA_PAGAMENTO            datetime64[us]
DATA_VENCIMENTO           datetime64[us]
VALOR_A_PAGAR                    float64
TAXA                             float64
DIAS_ATRASO                        int64
TARGET                             int64
DIAS_ATRASO_AJUSTADO               int64
ATRASO_BIN                      category
DATA_CADASTRO             datetime64[us]
DDD                                  str
SEGMENTO_INDUSTRIAL                  str
DOMINIO_EMAIL                        str
PORTE                                str
CEP_2_DIG                            str
RENDA_MES_ANTERIOR               float64
NO_FUNCIONARIOS                  float64
dtype: object

In [84]:
base_modelo["ANO_EMISSAO"] = base_modelo["DATA_EMISSAO_DOCUMENTO"].dt.year

base_modelo["MES_EMISSAO"] = base_modelo["DATA_EMISSAO_DOCUMENTO"].dt.month

base_modelo["ANO_CADASTRO"] = base_modelo["DATA_CADASTRO"].dt.year

base_modelo["TEMPO_CLIENTE"] = (
    base_modelo["DATA_EMISSAO_DOCUMENTO"] - base_modelo["DATA_CADASTRO"]
).dt.days

In [85]:
base_modelo[[
    "ANO_EMISSAO",
    "MES_EMISSAO",
    "ANO_CADASTRO",
    "TEMPO_CLIENTE"
]].head()

,ANO_EMISSAO,MES_EMISSAO,ANO_CADASTRO,TEMPO_CLIENTE
0,2018,8,2011,2741
1,2018,8,2011,2746
2,2018,8,2011,2747
3,2018,8,2011,2747
4,2018,8,2011,2748


In [86]:
import numpy as np

base_modelo["VALOR_LOG"] = np.log1p(base_modelo["VALOR_A_PAGAR"])

In [87]:
base_modelo[["VALOR_A_PAGAR", "VALOR_LOG"]].head()

,VALOR_A_PAGAR,VALOR_LOG
0,100616.1,11.519078
1,89552.8,11.402595
2,102686.1,11.539442
3,94062.8,11.451729
4,51393.0,10.847277


In [88]:
base_modelo["PAGOU_ATRASADO"] = (
    base_modelo["DIAS_ATRASO_AJUSTADO"] > 0
).astype(int)

In [89]:
base_modelo["PAGOU_ATRASADO"].value_counts()

PAGOU_ATRASADO
0    68648
1     8766
Name: count, dtype: int64

In [90]:
base_modelo["MES_CADASTRO"] = base_modelo["DATA_CADASTRO"].dt.month

In [91]:
base_modelo[["DATA_CADASTRO", "MES_CADASTRO"]].head()

,DATA_CADASTRO,MES_CADASTRO
0,2011-02-14,2
1,2011-02-14,2
2,2011-02-14,2
3,2011-02-14,2
4,2011-02-14,2


In [92]:
base_ml = base_modelo.copy()

In [93]:
base_ml = base_ml.drop(
    columns=[
        "ID_CLIENTE",
        "DATA_EMISSAO_DOCUMENTO",
        "DATA_PAGAMENTO",
        "DATA_VENCIMENTO",
        "DATA_CADASTRO",
        "DIAS_ATRASO",
        "DIAS_ATRASO_AJUSTADO",
        "PAGOU_ATRASADO"
    ]
)

In [94]:
base_ml.head()

,SAFRA_REF,VALOR_A_PAGAR,TAXA,TARGET,ATRASO_BIN,DDD,SEGMENTO_INDUSTRIAL,DOMINIO_EMAIL,PORTE,CEP_2_DIG,RENDA_MES_ANTERIOR,NO_FUNCIONARIOS,ANO_EMISSAO,MES_EMISSAO,ANO_CADASTRO,TEMPO_CLIENTE,VALOR_LOG,MES_CADASTRO
0,2018-08-01,100616.1,5.99,0,EM_DIA,11,Comércio,HOTMAIL,PEQUENO,27,240502.0,118.0,2018,8,2011,2741,11.519078,2
1,2018-08-01,89552.8,5.99,0,EM_DIA,11,Comércio,HOTMAIL,PEQUENO,27,240502.0,118.0,2018,8,2011,2746,11.402595,2
2,2018-08-01,102686.1,5.99,0,EM_DIA,11,Comércio,HOTMAIL,PEQUENO,27,240502.0,118.0,2018,8,2011,2747,11.539442,2
3,2018-08-01,94062.8,5.99,0,EM_DIA,11,Comércio,HOTMAIL,PEQUENO,27,240502.0,118.0,2018,8,2011,2747,11.451729,2
4,2018-08-01,51393.0,5.99,0,EM_DIA,11,Comércio,HOTMAIL,PEQUENO,27,240502.0,118.0,2018,8,2011,2748,10.847277,2


In [95]:
X = base_ml.drop("TARGET", axis=1)

y = base_ml["TARGET"]

In [96]:
print(X.shape)
print(y.shape)

(77414, 17)
(77414,)


In [97]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [98]:
print(X_train.shape)
print(X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(61931, 17)
(15483, 17)
TARGET
0    0.929777
1    0.070223
Name: proportion, dtype: float64
TARGET
0    0.929794
1    0.070206
Name: proportion, dtype: float64


In [99]:
cat_cols = X.select_dtypes(include=["str", "category"]).columns

cat_cols

Index(['ATRASO_BIN', 'DDD', 'SEGMENTO_INDUSTRIAL', 'DOMINIO_EMAIL', 'PORTE',
       'CEP_2_DIG'],
      dtype='str')

In [100]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [101]:
X_cat = encoder.fit_transform(X[cat_cols])

In [102]:
X_cat.shape

(77414, 190)

In [103]:
num_cols = X.select_dtypes(
    exclude=["str", "category"]
).columns

num_cols

Index(['SAFRA_REF', 'VALOR_A_PAGAR', 'TAXA', 'RENDA_MES_ANTERIOR',
       'NO_FUNCIONARIOS', 'ANO_EMISSAO', 'MES_EMISSAO', 'ANO_CADASTRO',
       'TEMPO_CLIENTE', 'VALOR_LOG', 'MES_CADASTRO'],
      dtype='str')

In [104]:
X_num = X[num_cols].values

In [105]:
import numpy as np

X_final = np.hstack([
    X_num,
    X_cat
])

In [106]:
X_final.shape

(77414, 201)

In [107]:
X_train_final, X_test_final, y_train, y_test = train_test_split(
    X_final,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [108]:
print(X_train_final.shape)
print(X_test_final.shape)

(61931, 201)
(15483, 201)


In [109]:
from sklearn.ensemble import RandomForestClassifier

modelo_rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

In [110]:
modelo_rf.fit(
    X_train_final,
    y_train
)

TypeError: float() argument must be a string or a real number, not 'Timestamp'

In [111]:
num_cols

Index(['SAFRA_REF', 'VALOR_A_PAGAR', 'TAXA', 'RENDA_MES_ANTERIOR',
       'NO_FUNCIONARIOS', 'ANO_EMISSAO', 'MES_EMISSAO', 'ANO_CADASTRO',
       'TEMPO_CLIENTE', 'VALOR_LOG', 'MES_CADASTRO'],
      dtype='str')

In [112]:
num_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns

num_cols

Index(['VALOR_A_PAGAR', 'TAXA', 'RENDA_MES_ANTERIOR', 'NO_FUNCIONARIOS',
       'TEMPO_CLIENTE', 'VALOR_LOG'],
      dtype='str')

In [113]:
X_num = X[num_cols].values

X_final = np.hstack([
    X_num,
    X_cat
])

print(X_final.shape)

(77414, 196)


In [114]:
X_train_final, X_test_final, y_train, y_test = train_test_split(
    X_final,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [115]:
modelo_rf.fit(
    X_train_final,
    y_train
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fe

In [116]:
y_pred = modelo_rf.predict(X_test_final)

In [117]:
y_pred[:20]

array([1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [118]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[14259   137]
 [   61  1026]]


In [119]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.99      0.99     14396
           1       0.88      0.94      0.91      1087

    accuracy                           0.99     15483
   macro avg       0.94      0.97      0.95     15483
weighted avg       0.99      0.99      0.99     15483



## Modelo 1 - Random Forest Baseline

Foi treinado um primeiro modelo utilizando Random Forest Classifier para prever a variável TARGET, que representa clientes com atraso igual ou superior a 5 dias.

### Configuração do modelo

- Algoritmo: Random Forest Classifier
- Número de árvores: 200
- Tratamento de desbalanceamento: class_weight="balanced"
- Divisão dos dados:
  - 80% treino
  - 20% teste
- Separação estratificada utilizando stratify=y

### Resultados

O modelo apresentou os seguintes resultados:

- Accuracy: 99%
- Precision da classe 1: 88%
- Recall da classe 1: 94%
- F1-score da classe 1: 91%

A matriz de confusão apresentou:

- Verdadeiros negativos: 14.259
- Falsos positivos: 137
- Falsos negativos: 61
- Verdadeiros positivos: 1.026

### Análise

O modelo apresentou boa capacidade de identificar clientes com risco de inadimplência, principalmente pelo recall de 94% da classe positiva.

Entretanto, foi identificada uma possível ocorrência de data leakage na variável ATRASO_BIN, pois essa variável foi criada a partir do DIAS_ATRASO, que também foi utilizado para gerar o TARGET.

Por esse motivo, será realizado um novo treinamento removendo essa variável para avaliar o desempenho do modelo utilizando apenas informações disponíveis antes do evento de inadimplência.

In [120]:
base_ml_sem_atraso = base_ml.drop(
    columns=["ATRASO_BIN"]
)

In [121]:
base_ml_sem_atraso.shape

(77414, 17)

In [122]:
X2 = base_ml_sem_atraso.drop("TARGET", axis=1)

y2 = base_ml_sem_atraso["TARGET"]

In [123]:
print(X2.shape)
print(y2.shape)

(77414, 16)
(77414,)


In [124]:
cat_cols2 = X2.select_dtypes(
    include=["str", "category"]
).columns

cat_cols2

Index(['DDD', 'SEGMENTO_INDUSTRIAL', 'DOMINIO_EMAIL', 'PORTE', 'CEP_2_DIG'], dtype='str')

In [125]:
from sklearn.preprocessing import OneHotEncoder

encoder2 = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [126]:
X_cat2 = encoder2.fit_transform(
    X2[cat_cols2]
)

In [127]:
X_cat2.shape

(77414, 184)

In [128]:
num_cols2 = X2.select_dtypes(
    include=["int64", "float64"]
).columns

num_cols2

Index(['VALOR_A_PAGAR', 'TAXA', 'RENDA_MES_ANTERIOR', 'NO_FUNCIONARIOS',
       'TEMPO_CLIENTE', 'VALOR_LOG'],
      dtype='str')

In [129]:
X_num2 = X2[num_cols2].values

In [130]:
X_final2 = np.hstack([
    X_num2,
    X_cat2
])

In [131]:
X_final2.shape

(77414, 190)

In [132]:
X_train_final2, X_test_final2, y_train2, y_test2 = train_test_split(
    X_final2,
    y2,
    test_size=0.2,
    random_state=42,
    stratify=y2
)

In [133]:
print(X_train_final2.shape)
print(X_test_final2.shape)

(61931, 190)
(15483, 190)


In [134]:
modelo_rf2 = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

In [135]:
modelo_rf2.fit(
    X_train_final2,
    y_train2
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fe

In [136]:
y_pred2 = modelo_rf2.predict(
    X_test_final2
)

In [137]:
y_pred2[:20]

array([1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [138]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test2, y_pred2))

[[13964   432]
 [  245   842]]


In [139]:
print(classification_report(y_test2, y_pred2))

              precision    recall  f1-score   support

           0       0.98      0.97      0.98     14396
           1       0.66      0.77      0.71      1087

    accuracy                           0.96     15483
   macro avg       0.82      0.87      0.84     15483
weighted avg       0.96      0.96      0.96     15483



## Modelo 2 - Random Forest sem variável de vazamento

Após identificar uma possível ocorrência de data leakage na variável ATRASO_BIN, foi realizado um novo treinamento removendo essa feature.

O objetivo foi avaliar o desempenho do modelo utilizando apenas informações disponíveis antes do evento de inadimplência.

### Resultados

O modelo apresentou:

- Accuracy: 96%
- Precision (classe 1): 66%
- Recall (classe 1): 77%
- F1-score (classe 1): 71%

Matriz de confusão:

- Verdadeiros negativos: 13.964
- Falsos positivos: 432
- Falsos negativos: 245
- Verdadeiros positivos: 842

### Análise

O desempenho reduziu em relação ao modelo inicial, confirmando que ATRASO_BIN possuía forte poder preditivo por conter informação derivada diretamente do comportamento de atraso.

Apesar da redução das métricas, o segundo modelo representa melhor um cenário real de previsão de risco, pois utiliza somente informações cadastrais e financeiras disponíveis antes da ocorrência do atraso.